In [ ]:
pip install -U "bitsandbytes>=0.46.1"

### charger le model Qwen2.5-Math-7B-Instruct

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-Math-7B-Instruct"

quant_config = BitsAndBytesConfig(load_in_4bit=True)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
)

### charger le modele Qwen2.5-7B-Instruct

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
model_name = "Qwen/Qwen2.5-7B-Instruct"

quant_config = BitsAndBytesConfig(load_in_4bit=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto"
)

### prompt 2nd version 

In [ ]:
system_prompt = """You are a mathematics expert tasked with evaluating whether a given list of hypotheses for a theorem is correct — i.e., necessary and sufficient for the theorem to be applicable.

EVALUATION RULES:
1. A theorem has a set of hypotheses that are truly NECESSARY AND SUFFICIENT for it to be applicable (including, where relevant, problem data such as an initial condition, a domain of definition, a reference point, etc.).
2. Compare the proposed list of hypotheses to this actual required set.
3. A proposed hypothesis that is STRONGER than the actually required one is ACCEPTED, provided it logically implies it AND it REPLACES the required hypothesis (rather than being added alongside it).
   Example: if the theorem requires "f continuous" and the list REPLACES this with "f differentiable", this is CORRECT because differentiable ⇒ continuous, and no separate "f continuous" hypothesis is also present.
4. IMPORTANT — Redundant additions are NOT acceptable, even if logically valid: if the list already contains a hypothesis sufficient to satisfy a requirement, and an ADDITIONAL hypothesis is included that is merely implied by (redundant with) one already in the list, this extra hypothesis must be flagged as an invented hypothesis. Being mathematically true or logically implied does NOT make a hypothesis necessary in the list.
   Example: if the list contains "f is Lipschitz in y" (already sufficient) AND ALSO "f is continuous in y" (implied by the first), the second is redundant and must be flagged as "invented_hypothesis" — do NOT mark the list as correct just because the added hypothesis happens to be true.
5. Only take into account the hypotheses listed. Do not assume implicit hypotheses that are not mentioned as being "understood".
6. Do not discuss the mathematical truth of the theorem itself, only the completeness and correctness of the provided hypothesis list relative to the standard version of the theorem.

DECISION PROCEDURE (apply in this order):
a) List the actual necessary-and-sufficient hypotheses for the standard version of the theorem.
b) Check if every proposed hypothesis maps to a necessary one, either directly or as a valid REPLACEMENT (stronger version implying it, per rule 3).
c) Check if any proposed hypothesis is NOT needed to satisfy any requirement — i.e., it is redundant with another hypothesis already in the list, or entirely unrelated to the theorem. If so → "invented_hypothesis", regardless of whether it is mathematically true.
d) Check if any necessary hypothesis has no corresponding entry in the list at all → missing.
e) Only if every necessary hypothesis is covered exactly once (directly or via valid replacement) AND no extra/redundant hypothesis is present → label = "correct".

If the hypotheses are necessary and sufficient, the label is "correct" and error_type must be null.

If the hypotheses are NOT correct, the label is "incorrect" and error_type must be exactly one of the following categories (choose the single most fitting one):
- "missing_hypothesis": exactly one necessary hypothesis is missing.
- "multiple_missing": more than one necessary hypothesis is missing.
- "misformulated_hypothesis": a hypothesis is present but stated imprecisely or incorrectly (e.g., wrong quantifier, wrong domain, wrong scope) so that it does not actually match what is required.
- "invented_hypothesis": an extra hypothesis is added that is not necessary for the theorem — including hypotheses that are redundant with (implied by) another hypothesis already present in the list, even if mathematically true.
- "erroneous_interval": a hypothesis involves an interval/domain/set that is wrong (too large, too small, open instead of closed, etc.).
- "invalid_implication": a stronger hypothesis is proposed as a replacement for a required one, but it does NOT actually logically imply the required hypothesis.

OUTPUT FORMAT (strict JSON, nothing else, no markdown, no explanation outside the JSON):
{
  "label": "correct" or "incorrect",
  "error_type": null or one of ["missing_hypothesis", "multiple_missing", "misformulated_hypothesis", "invented_hypothesis", "erroneous_interval", "invalid_implication"],
  "reason": "concise explanation of why the hypotheses are correct/sufficient, or which specific hypothesis is missing/wrong/extra and why, including relevant logical implications"
}
"""

user_prompt = """Theorem: Cauchy-Lipschitz Theorem

Proposed hypotheses:
 "f is continuous in t",
      "f is Lipschitz in y"

Evaluate whether this list of hypotheses is necessary and sufficient for the application of this theorem, according to the rules and decision procedure given in the system prompt. Respond only in the requested JSON format."""

### Evaluation for one example


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig



tokenizer = AutoTokenizer.from_pretrained(model_name)


messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=1024,
    temperature=0.7,
    do_sample=True,
    top_p=0.9,
    pad_token_id=tokenizer.eos_token_id,
)

response = tokenizer.decode(
    
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)
print(response)

### Evaluation for the entire dataset


In [ ]:
"""
Benchmark de vérification d'hypothèses de théorèmes mathématiques
avec Qwen2.5-7B-Instruct (4-bit) sur Kaggle.

Le fichier JSON doit contenir une liste d'objets du type :

{
  "theorem_id": "T21",
  "name": "Adjacent Sequences Theorem",
  "copy": [...],
  "expected": [...],
  "is_correct": false,
  "reason": "...",
  "error_type": "missing_hypothesis"
}
"""

import json
import re
import time
import torch
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)


# Adapter selon ton dataset Kaggle

INPUT_JSON_PATH = "/kaggle/input/datasets/mathurasanthalingam/benchmark-data-json/benchmark_data.json"
OUTPUT_CSV_PATH = "/kaggle/working/benchmark_results.csv"

# ============================================================
# CHARGEMENT DU MODELE
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(model_name)



# ============================================================
# PROMPTS
# ============================================================
SYSTEM_PROMPT = """You are a mathematics expert tasked with evaluating whether a given list of hypotheses for a theorem is correct — i.e., necessary and sufficient for the theorem to be applicable.

EVALUATION RULES:
1. A theorem has a set of hypotheses that are truly NECESSARY AND SUFFICIENT for it to be applicable (including, where relevant, problem data such as an initial condition, a domain of definition, a reference point, etc.).
2. Compare the proposed list of hypotheses to this actual required set.
3. A proposed hypothesis that is STRONGER than the actually required one is ACCEPTED, provided it logically implies it AND it REPLACES the required hypothesis (rather than being added alongside it).
   Example: if the theorem requires "f continuous" and the list REPLACES this with "f differentiable", this is CORRECT because differentiable ⇒ continuous, and no separate "f continuous" hypothesis is also present.
4. IMPORTANT — Redundant additions are NOT acceptable, even if logically valid: if the list already contains a hypothesis sufficient to satisfy a requirement, and an ADDITIONAL hypothesis is included that is merely implied by (redundant with) one already in the list, this extra hypothesis must be flagged as an invented hypothesis. Being mathematically true or logically implied does NOT make a hypothesis necessary in the list.
   Example: if the list contains "f is Lipschitz in y" (already sufficient) AND ALSO "f is continuous in y" (implied by the first), the second is redundant and must be flagged as "invented_hypothesis" — do NOT mark the list as correct just because the added hypothesis happens to be true.
5. Only take into account the hypotheses listed. Do not assume implicit hypotheses that are not mentioned as being "understood".
6. Do not discuss the mathematical truth of the theorem itself, only the completeness and correctness of the provided hypothesis list relative to the standard version of the theorem.

DECISION PROCEDURE (apply in this order):
a) List the actual necessary-and-sufficient hypotheses for the standard version of the theorem.
b) Check if every proposed hypothesis maps to a necessary one, either directly or as a valid REPLACEMENT (stronger version implying it, per rule 3).
c) Check if any proposed hypothesis is NOT needed to satisfy any requirement — i.e., it is redundant with another hypothesis already in the list, or entirely unrelated to the theorem. If so → "invented_hypothesis", regardless of whether it is mathematically true.
d) Check if any necessary hypothesis has no corresponding entry in the list at all → missing.
e) Only if every necessary hypothesis is covered exactly once (directly or via valid replacement) AND no extra/redundant hypothesis is present → label = "correct".

If the hypotheses are necessary and sufficient, the label is "correct" and error_type must be null.

If the hypotheses are NOT correct, the label is "incorrect" and error_type must be exactly one of the following categories (choose the single most fitting one):
- "missing_hypothesis": exactly one necessary hypothesis is missing.
- "multiple_missing": more than one necessary hypothesis is missing.
- "invented_hypothesis": a hypothesis that is not necessary for the theorem — this covers a hypothesis stated imprecisely or incorrectly (wrong quantifier, wrong domain, wrong scope) so that it does not match what is required, an extra hypothesis added that is not necessary for the theorem, or a redundant hypothesis merely implied by another one already present.

OUTPUT FORMAT (strict JSON, nothing else, no markdown, no explanation outside the JSON):
{
  "label": "correct" or "incorrect",
  "error_type": null or one of ["missing_hypothesis", "multiple_missing", "invented_hypothesis"],
  "reason": "concise explanation of why the hypotheses are correct/sufficient, or which specific hypothesis is missing/wrong/extra and why, including relevant logical implications"
}
"""

USER_PROMPT_TEMPLATE = """Theorem: {theorem_name}

Proposed hypotheses:
{hypotheses_list}

Evaluate whether this list is necessary and sufficient.

Return ONLY the JSON.
"""


def build_user_prompt(theorem_name, hypotheses):
    hyp = "\n".join(f"- {h}" for h in hypotheses)

    return USER_PROMPT_TEMPLATE.format(
        theorem_name=theorem_name,
        hypotheses_list=hyp,
    )


# ============================================================
# APPEL LLM
# ============================================================

def call_llm(theorem_name, hypotheses):

    user_prompt = build_user_prompt(
        theorem_name,
        hypotheses,
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=1000,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True,
    )

    return answer.strip()


# ============================================================
# PARSER JSON
# ============================================================

def parse_llm_output(raw_text):

    if raw_text is None:
        return {
            "label": None,
            "error_type": None,
            "reason": None,
            "parse_error": "no_response",
        }

    cleaned = re.sub(
        r"```json|```",
        "",
        raw_text,
        flags=re.IGNORECASE,
    ).strip()

    try:

        parsed = json.loads(cleaned)

        return {
            "label": parsed.get("label"),
            "error_type": parsed.get("error_type"),
            "reason": parsed.get("reason"),
            "parse_error": None,
        }

    except Exception as e:

        return {
            "label": None,
            "error_type": None,
            "reason": None,
            "parse_error": str(e),
        }


# ============================================================
# CHARGEMENT DATASET
# ============================================================

print("Loading dataset...")

with open(INPUT_JSON_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

print(f"{len(dataset)} examples loaded.")

# ============================================================
# BENCHMARK
# ============================================================

results = []

start = time.time()

for i, sample in enumerate(dataset):

    theorem_name = sample["name"]
    hypotheses = sample["copy"]

    try:

        raw = call_llm(
            theorem_name,
            hypotheses,
        )

        parsed = parse_llm_output(raw)

    except Exception as e:

        raw = None

        parsed = {
            "label": None,
            "error_type": None,
            "reason": None,
            "parse_error": str(e),
        }

    results.append({

        "theorem_id": sample["theorem_id"],

        "name": theorem_name,

        "hypotheses": sample["copy"],

        "expected": sample["expected"],

        "ground_truth_label":
            "correct" if sample["is_correct"] else "incorrect",

        "ground_truth_error":
            sample["error_type"],

        "ground_truth_reason":
            sample["reason"],

        "prediction_label":
            parsed["label"],

        "prediction_error":
            parsed["error_type"],

        "prediction_reason":
            parsed["reason"],

        "parse_error":
            parsed["parse_error"],

        "raw_response":
            raw,
    })

    if (i + 1) % 10 == 0:
        print(f"{i+1}/{len(dataset)} examples processed")

# ============================================================
# SAUVEGARDE
# ============================================================

df = pd.DataFrame(results)

df.to_csv(
    OUTPUT_CSV_PATH,
    index=False,
)

print()
print("CSV saved to:")
print(OUTPUT_CSV_PATH)

# ============================================================
# METRIQUES
# ============================================================

valid = df[df["parse_error"].isna()].copy()

label_accuracy = (
    valid["ground_truth_label"]
    ==
    valid["prediction_label"]
).mean()

error_accuracy = (
    valid["ground_truth_error"].fillna("none")
    ==
    valid["prediction_error"].fillna("none")
).mean()

print()
print("=" * 60)
print("RESULTS")
print("=" * 60)
print(f"Examples           : {len(df)}")
print(f"Valid generations  : {len(valid)}")
print(f"Label accuracy     : {label_accuracy:.4f}")
print(f"Error accuracy     : {error_accuracy:.4f}")
print(f"Execution time     : {(time.time()-start)/60:.2f} min")

display(df.head())

In [ ]:
import pandas as pd
import json
import re

# 1. On recharge le tableau depuis le fichier sauvegardé sur le disque
df = pd.read_csv("/kaggle/working/benchmark_results.csv")

# 2. On définit la fonction de nettoyage
def extract_json_robust(raw_text):
    if pd.isna(raw_text):
        return None, None, None, "no_text"
    
    # Cette expression régulière isole tout ce qui se trouve entre la première '{' et la dernière '}'
    match = re.search(r'\{.*\}', str(raw_text), re.DOTALL)
    
    if match:
        json_str = match.group(0)
        try:
            parsed = json.loads(json_str)
            return parsed.get("label"), parsed.get("error_type"), parsed.get("reason"), None
        except Exception as e:
            return None, None, None, f"invalid_json: {e}"
    else:
        return None, None, None, "no_json_brackets_found"

# 3. On applique la correction
print("Extraction des JSON en cours...")
for idx, row in df.iterrows():
    l, e, r, err = extract_json_robust(row["raw_response"])
    df.loc[idx, "prediction_label"] = l
    df.loc[idx, "prediction_error"] = e
    df.loc[idx, "prediction_reason"] = r
    df.loc[idx, "parse_error"] = err

# 4. On sauvegarde la version corrigée
df.to_csv("/kaggle/working/benchmark_results_reparsed.csv", index=False)

# 5. On calcule les vrais scores
valid = df[df["parse_error"].isna()]
label_acc = (valid["ground_truth_label"] == valid["prediction_label"]).mean()
error_acc = (valid["ground_truth_error"].fillna("none") == valid["prediction_error"].fillna("none")).mean()

print("\n" + "=" * 60)
print("NOUVEAUX RÉSULTATS (Après réparation du parsing)")
print("=" * 60)
print(f"Valid generations : {len(valid)} / {len(df)}")
print(f"Label accuracy    : {label_acc:.4f}")
print(f"Error accuracy    : {error_acc:.4f}")

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

### Si y'a des lignes qui n'ont pas pu etre parsée

In [ ]:
import pandas as pd
import time

# ============================================================
# PROMPTS
# ============================================================
SYSTEM_PROMPT = """You are a mathematics expert tasked with evaluating whether a given list of hypotheses for a theorem is correct — i.e., necessary and sufficient for the theorem to be applicable.

EVALUATION RULES:
1. A theorem has a set of hypotheses that are truly NECESSARY AND SUFFICIENT for it to be applicable (including, where relevant, problem data such as an initial condition, a domain of definition, a reference point, etc.).
2. Compare the proposed list of hypotheses to this actual required set.
3. A proposed hypothesis that is STRONGER than the actually required one is ACCEPTED, provided it logically implies it AND it REPLACES the required hypothesis (rather than being added alongside it).
   Example: if the theorem requires "f continuous" and the list REPLACES this with "f differentiable", this is CORRECT because differentiable ⇒ continuous, and no separate "f continuous" hypothesis is also present.
4. IMPORTANT — Redundant additions are NOT acceptable, even if logically valid: if the list already contains a hypothesis sufficient to satisfy a requirement, and an ADDITIONAL hypothesis is included that is merely implied by (redundant with) one already in the list, this extra hypothesis must be flagged as an invented/unnecessary hypothesis. Being mathematically true or logically implied does NOT make a hypothesis necessary in the list.
   Example: if the list contains "f is Lipschitz in y" (already sufficient) AND ALSO "f is continuous in y" (implied by the first), the second is redundant and must be flagged as "invented_hypothesis" — do NOT mark the list as correct just because the added hypothesis happens to be true.
5. Only take into account the hypotheses listed. Do not assume implicit hypotheses that are not mentioned as being "understood".
6. Do not discuss the mathematical truth of the theorem itself, only the completeness and correctness of the provided hypothesis list relative to the standard version of the theorem.

DECISION PROCEDURE (apply in this order):
a) List the actual necessary-and-sufficient hypotheses for the standard version of the theorem.
b) Check if every proposed hypothesis maps to a necessary one, either directly or as a valid REPLACEMENT (stronger version implying it, per rule 3).
c) Check if any proposed hypothesis is NOT needed to satisfy any requirement — i.e., it is redundant with another hypothesis already in the list, or entirely unrelated to the theorem. If so → "invented_hypothesis", regardless of whether it is mathematically true.
d) Check if any necessary hypothesis has no corresponding entry in the list at all → missing.
e) Only if every necessary hypothesis is covered exactly once (directly or via valid replacement) AND no extra/redundant hypothesis is present → label = "correct".

If the hypotheses are necessary and sufficient, the label is "correct" and error_type must be null.

If the hypotheses are NOT correct, the label is "incorrect" and error_type must be exactly one of the following categories (choose the single most fitting one):
- "missing_hypothesis": exactly one necessary hypothesis is missing.
- "multiple_missing": more than one necessary hypothesis is missing.
- "invented_hypothesis": a hypothesis that is not necessary for the theorem, it can be a hypothesis is present but stated imprecisely or incorrectly (e.g., wrong quantifier, wrong domain, wrong scope) so that it does not actually match what is required, or a  an extra hypothesis is added that is not necessary for the theorem. 


OUTPUT FORMAT (strict JSON, nothing else, no markdown, no explanation outside the JSON):
{
  "label": "correct" or "incorrect",
  "error_type": null or one of ["missing_hypothesis", "multiple_missing", "invented_hypothesis"],
  "reason": "concise explanation of why the hypotheses are correct/sufficient, or which specific hypothesis is missing/wrong/extra and why, including relevant logical implications"
}
"""

USER_PROMPT_TEMPLATE = """Theorem: {theorem_name}

Proposed hypotheses:
{hypotheses_list}

Evaluate whether this list is necessary and sufficient.

Return ONLY the JSON.
"""


# ============================================================
# PARSER JSON
# ============================================================

def parse_llm_output(raw_text):

    if raw_text is None:
        return {
            "label": None,
            "error_type": None,
            "reason": None,
            "parse_error": "no_response",
        }

    cleaned = re.sub(
        r"```json|```",
        "",
        raw_text,
        flags=re.IGNORECASE,
    ).strip()

    try:

        parsed = json.loads(cleaned)

        return {
            "label": parsed.get("label"),
            "error_type": parsed.get("error_type"),
            "reason": parsed.get("reason"),
            "parse_error": None,
        }

    except Exception as e:

        return {
            "label": None,
            "error_type": None,
            "reason": None,
            "parse_error": str(e),
        }






def build_user_prompt(theorem_name, hypotheses):
    hyp = "\n".join(f"- {h}" for h in hypotheses)

    return USER_PROMPT_TEMPLATE.format(
        theorem_name=theorem_name,
        hypotheses_list=hyp,
    )



def call_llm(theorem_name, hypotheses):

    user_prompt = build_user_prompt(
        theorem_name,
        hypotheses,
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True,
    )

    return answer.strip()

# df doit être ton DataFrame en mémoire (celui produit par le run précédent)
# Si tu repars d'un CSV sauvegardé, décommente la ligne suivante :
df = pd.read_csv("/kaggle/working/benchmark_results.csv")

MAX_RETRIES_PER_ROW = 3  # nombre de tentatives de relance par ligne avant abandon

failed_mask = df["parse_error"].notna()
failed_indices = df[failed_mask].index

print(f"{len(failed_indices)} lignes à ré-exécuter.")

for idx in failed_indices:
    row = df.loc[idx]
    theorem_name = row["name"]

    # "hypotheses" peut être une liste (si df encore en mémoire) ou une string (si relu depuis CSV)
    hyps = row["hypotheses"]
    if isinstance(hyps, str):
        import ast
        hyps = ast.literal_eval(hyps)

    print(f"\nRé-exécution : {row['theorem_id']} - {theorem_name}")

    success = False
    for attempt in range(1, MAX_RETRIES_PER_ROW + 1):
        try:
            raw = call_llm(theorem_name, hyps)
            parsed = parse_llm_output(raw)

            if parsed["parse_error"] is None:
                # Succès : on met à jour la ligne dans df
                df.loc[idx, "prediction_label"] = parsed["label"]
                df.loc[idx, "prediction_error"] = parsed["error_type"]
                df.loc[idx, "prediction_reason"] = parsed["reason"]
                df.loc[idx, "parse_error"] = None
                df.loc[idx, "raw_response"] = raw
                print(f"  -> Succès (tentative {attempt})")
                success = True
                break
            else:
                print(f"  -> Tentative {attempt} échouée (parse_error: {parsed['parse_error']})")

        except Exception as e:
            print(f"  -> Tentative {attempt} : erreur d'exécution ({e})")

        time.sleep(1)

    if not success:
        print(f"  -> ABANDON après {MAX_RETRIES_PER_ROW} tentatives pour {row['theorem_id']}")

# --- Sauvegarde du DataFrame mis à jour ---
OUTPUT_CSV_PATH = "/kaggle/working/benchmark_results.csv"
df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"\nDataFrame mis à jour sauvegardé dans {OUTPUT_CSV_PATH}")

# --- Vérification finale ---
remaining_failed = df["parse_error"].notna().sum()
print(f"Lignes encore en échec après ré-exécution : {remaining_failed}")